# P1T exploratory: how large would reporting noise have to be?

Calibrates a no-feedback, noisy-report economy and a feedback economy to Taiwan **development-period** data (2013–2016 only) and computes the forecast gain each would produce. No validation or holdout flows are read and no P1T statistic is computed on data. Run the single code cell once (a few minutes) and return `p1t_confound_sizing.zip`.

In [ ]:

import os, io, json, gzip, time, zipfile, hashlib, datetime, re, sys
import numpy as np, requests
EXECUTOR_SHA256 = "529a26c7f7ff49def539da1e108f199249f1f8f515a307b8f3c93ff801255a17"
EXECUTOR_SRC = "\"\"\"P1T executor: Taiwan tagged-flow forecast-gain test (registered spec P1T_taiwan_spec.md).\n\nLayers\n------\n1. Core (pure numpy): per-stock forecast losses, the primary statistic D, and the stationary date bootstrap.\n2. Data: parse stored TWSE T86 and MI_INDEX JSON responses into a panel, apply the universe and exclusion rules,\n   and build u (class flows), x (market-reference basis) and the limit indicator.\n3. Run: one period at a time (\"validation\" or \"holdout\"), writing a results JSON.\n\nThe executor never downloads. Downloads are done by the Colab notebook, which stores raw responses with SHA-256.\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport re\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nD_STAR = 0.0035\nSEED = 20261107\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 1. Core\n# ----------------------------------------------------------------------------------------------------------------\n\ndef _design(u, x, lim, rows, with_x):\n    \"\"\"Rows index the forecast origin t; target is u[t+1].\"\"\"\n    cols = [np.ones(len(rows)), u[rows], u[rows - 1], lim[rows]]\n    if with_x:\n        cols += [x[rows], x[rows - 1]]\n    return np.column_stack(cols)\n\n\ndef stock_losses(u, x, lim, usable, fit_mask, eval_mask):\n    \"\"\"Per-date loss differential for one stock.\n\n    u, x, lim, usable: arrays over the full date axis. usable[t] means day t may serve as a forecast origin\n    (t-1, t, t+1 valid and x_t, x_{t-1} defined). fit_mask / eval_mask select origins for fitting / evaluation.\n    Returns (ell, var_fit) where ell[t] = (e0^2 - e1^2) / var_fit on evaluation origins and NaN elsewhere.\n    \"\"\"\n    T = len(u)\n    t_all = np.arange(1, T - 1)\n    fin = np.isfinite(u)\n    # Amendment A1: an origin needs finite flows at t-1, t, t+1 (a no-op for Foreign; needed for classes that a\n    # T86 schema era does not report).\n    ok = usable[1:T - 1] & fin[0:T - 2] & fin[1:T - 1] & fin[2:T]\n    fit = t_all[ok & fit_mask[1:T - 1]]\n    ev = t_all[ok & eval_mask[1:T - 1]]\n    ell = np.full(T, np.nan)\n    if len(fit) < 50 or len(ev) < 50:\n        return ell, np.nan\n    y_fit = u[fit + 1]\n    var_fit = float(np.var(y_fit))\n    if not np.isfinite(var_fit) or var_fit <= 0:\n        return ell, np.nan\n    b0 = np.linalg.lstsq(_design(u, x, lim, fit, False), y_fit, rcond=None)[0]\n    b1 = np.linalg.lstsq(_design(u, x, lim, fit, True), y_fit, rcond=None)[0]\n    e0 = u[ev + 1] - _design(u, x, lim, ev, False) @ b0\n    e1 = u[ev + 1] - _design(u, x, lim, ev, True) @ b1\n    ell[ev] = (e0 ** 2 - e1 ** 2) / var_fit\n    return ell, var_fit\n\n\ndef panel_D(L):\n    \"\"\"L: stocks x dates, NaN where not an evaluation origin. D = mean over stocks of each stock's mean loss gain.\"\"\"\n    per = np.nanmean(L, axis=1)\n    per = per[np.isfinite(per)]\n    return float(per.mean()), per\n\n\ndef stationary_indices(n, mean_block, rng):\n    p = 1.0 / mean_block\n    idx = np.empty(n, dtype=np.int64)\n    idx[0] = rng.integers(n)\n    for i in range(1, n):\n        idx[i] = rng.integers(n) if rng.random() < p else (idx[i - 1] + 1) % n\n    return idx\n\n\ndef bootstrap_D(L, mean_block=20, B=9999, seed=SEED):\n    \"\"\"Stationary bootstrap over dates, the same resampled dates for every stock.\"\"\"\n    cols = np.where(np.isfinite(L).any(axis=0))[0]\n    Lc = L[:, cols]\n    rng = np.random.default_rng(seed)\n    n = Lc.shape[1]\n    S = np.nan_to_num(Lc)\n    C = np.isfinite(Lc).astype(float)\n    out = np.empty(B)\n    for b in range(B):\n        idx = stationary_indices(n, mean_block, rng)\n        num = S[:, idx].sum(1)\n        cnt = C[:, idx].sum(1)\n        per = num[cnt > 0] / cnt[cnt > 0]\n        out[b] = per.mean()\n    return out\n\n\ndef decide(D, boot):\n    \"\"\"Registered criteria. One-sided p for D > 0 from the bootstrap distribution centred at D.\"\"\"\n    lo, hi = np.percentile(boot, [2.5, 97.5])\n    p = float(np.mean(boot - D >= D)) if D > 0 else 1.0\n    if p < 0.05 and D > 0 and lo > 0:\n        verdict = \"supportive\"\n    elif hi < D_STAR:\n        verdict = \"inconsistent\"\n    else:\n        verdict = \"inconclusive\"\n    return {\"D\": D, \"p_one_sided\": p, \"ci95\": [float(lo), float(hi)], \"D_star\": D_STAR, \"verdict\": verdict}\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 2. Data\n# ----------------------------------------------------------------------------------------------------------------\n\n# Amendment A1 (7 November 2026, before any statistic was computed). TWSE's T86 report has three schema eras in the\n# registered sample (census of every stored response, schema_census.json):\n#   E1 2013-01-02 .. 2014-11-28: 外資買賣超股數, 投信買賣超股數, 自營商買賣超股數 (no own/hedging split)\n#   E2 2014-12-01 .. 2017-12-15: as E1 plus 自營商買賣超股數(自行買賣) and 自營商買賣超股數(避險)\n#   E3 2017-12-18 ..           : 外陸資買賣超股數(不含外資自營商) and 外資自營商買賣超股數 replace 外資買賣超股數\n# Each series is read from the first name in its list that the day's schema contains. Foreign is the reported foreign\n# total in E1-E2 and the sum of its two parts in E3 (spec: \"Foreign (including foreign dealers)\"). A series that a\n# day's schema does not report is NaN for every stock that day (never zero). An unknown schema stops the run.\nT86_FIELDS = {\n    \"foreign_ex\": [\"外陸資買賣超股數(不含外資自營商)\", \"外資買賣超股數\"],\n    \"foreign_dealer\": [\"外資自營商買賣超股數\", None],     # None: contained in the E1-E2 foreign total, read as 0\n    \"trust\": [\"投信買賣超股數\"],\n    \"dealer_total\": [\"自營商買賣超股數\"],\n    \"dealer_own\": [\"自營商買賣超股數(自行買賣)\"],\n    \"dealer_hedge\": [\"自營商買賣超股數(避險)\"],\n    \"total3\": [\"三大法人買賣超股數\"],\n}\nREQUIRED = (\"foreign_ex\", \"trust\", \"dealer_total\")\nQUOTE_TITLE_SUFFIX = \"權證、牛熊證、可展延牛熊證))\"\nINDEX_NAME = \"發行量加權股價指數\"\n\n\ndef _num(s):\n    s = str(s).replace(\",\", \"\").strip()\n    if s in (\"\", \"--\", \"---\", \"X\"):\n        return math.nan\n    try:\n        return float(s)\n    except ValueError:\n        return math.nan\n\n\ndef t86_positions(fields):\n    \"\"\"Column index per series for one schema, None if the series is contained elsewhere (read as 0), -1 if absent.\"\"\"\n    pos = {}\n    for k, names in T86_FIELDS.items():\n        hit = next((n for n in names if n is not None and n in fields), \"absent\")\n        if hit == \"absent\" and None in names and any(n in fields for n in T86_FIELDS[\"foreign_ex\"][1:]):\n            pos[k] = None\n        else:\n            pos[k] = fields.index(hit) if hit != \"absent\" else -1\n    for k in REQUIRED:\n        if pos[k] is None or pos[k] < 0:\n            raise ValueError(f\"unknown T86 schema: required series {k} not found in {fields}\")\n    return pos\n\n\ndef parse_t86(j):\n    \"\"\"Returns (records, unavailable): records[code][series] and the set of series this day's schema lacks.\"\"\"\n    pos = t86_positions(j[\"fields\"])\n    unavailable = {k for k, i in pos.items() if i is not None and i < 0}\n    out = {}\n    for row in j.get(\"data\", []):\n        code = row[0].strip()\n        if len(code) == 4 and code.isdigit():\n            out[code] = {k: (0.0 if i is None else (math.nan if i < 0 else _num(row[i]))) for k, i in pos.items()}\n    return out, unavailable\n\n\ndef t86_diagnostics(day_files):\n    \"\"\"Flow-only accounting checks per schema era (no prices, no forecasts). Shares of stock-days on which the\n    three-institution total equals the sum of the parts, with and without the foreign dealers, and on which the dealer\n    total equals own + hedging; and the foreign dealers' share of absolute foreign net flow.\"\"\"\n    eras = {}\n    for d, t86, _ in day_files:\n        key = tuple(t86[\"fields\"])\n        e = eras.setdefault(key, {\"first\": d, \"last\": d, \"days\": 0, \"rows\": 0, \"total_with_fd\": 0, \"total_without_fd\": 0,\n                                  \"dealer_rows\": 0, \"dealer_sum_ok\": 0, \"abs_fd\": 0.0, \"abs_foreign\": 0.0})\n        e[\"last\"] = d; e[\"days\"] += 1\n        rec, unavail = parse_t86(t86)\n        for r in rec.values():\n            vals = [r[\"foreign_ex\"], r[\"foreign_dealer\"], r[\"trust\"], r[\"dealer_total\"], r[\"total3\"]]\n            if not all(np.isfinite(v) for v in vals):\n                continue\n            e[\"rows\"] += 1\n            e[\"total_with_fd\"] += abs(r[\"total3\"] - (r[\"foreign_ex\"] + r[\"foreign_dealer\"] + r[\"trust\"] + r[\"dealer_total\"])) < 0.5\n            e[\"total_without_fd\"] += abs(r[\"total3\"] - (r[\"foreign_ex\"] + r[\"trust\"] + r[\"dealer_total\"])) < 0.5\n            if np.isfinite(r[\"dealer_own\"]) and np.isfinite(r[\"dealer_hedge\"]):\n                e[\"dealer_rows\"] += 1\n                e[\"dealer_sum_ok\"] += abs(r[\"dealer_total\"] - r[\"dealer_own\"] - r[\"dealer_hedge\"]) < 0.5\n            e[\"abs_fd\"] += abs(r[\"foreign_dealer\"]); e[\"abs_foreign\"] += abs(r[\"foreign_ex\"] + r[\"foreign_dealer\"])\n    out = []\n    for key, e in eras.items():\n        n = max(e[\"rows\"], 1)\n        out.append({\"first\": e[\"first\"], \"last\": e[\"last\"], \"days\": e[\"days\"], \"rows\": e[\"rows\"],\n                    \"share_total_eq_parts_with_foreign_dealers\": e[\"total_with_fd\"] / n,\n                    \"share_total_eq_parts_without_foreign_dealers\": e[\"total_without_fd\"] / n,\n                    \"share_dealer_total_eq_own_plus_hedge\": (e[\"dealer_sum_ok\"] / e[\"dealer_rows\"]) if e[\"dealer_rows\"] else None,\n                    \"foreign_dealer_share_of_abs_foreign_flow\": e[\"abs_fd\"] / e[\"abs_foreign\"] if e[\"abs_foreign\"] else None,\n                    \"fields\": list(key)})\n    return out\n\n\ndef parse_mi_index(j):\n    quotes, index = {}, math.nan\n    for t in j.get(\"tables\", []):\n        title = t.get(\"title\") or \"\"\n        fields = t.get(\"fields\") or []\n        if title.endswith(QUOTE_TITLE_SUFFIX) and \"證券代號\" in fields:\n            ic, iv, ival = fields.index(\"證券代號\"), fields.index(\"成交股數\"), fields.index(\"成交金額\")\n            icl, isg = fields.index(\"收盤價\"), fields.index(\"漲跌(+/-)\")\n            for row in t.get(\"data\", []):\n                code = row[ic].strip()\n                if len(code) == 4 and code.isdigit():\n                    sign = re.sub(r\"<[^>]+>\", \"\", str(row[isg])).strip()\n                    quotes[code] = {\"vol\": _num(row[iv]), \"value\": _num(row[ival]), \"close\": _num(row[icl]),\n                                    \"event\": sign == \"X\"}\n        if fields and t.get(\"data\") and any(str(r[0]).strip() == INDEX_NAME for r in t[\"data\"]):\n            for r in t[\"data\"]:\n                if str(r[0]).strip() == INDEX_NAME:\n                    index = _num(r[1])\n    return quotes, index\n\n\ndef parse_twt49u(j):\n    \"\"\"Ex-rights / ex-dividend calculation table. Returns a set of (YYYYMMDD, code). Field names are matched by\n    content because probe 2 returned no rows; unparsed tables are reported, not guessed.\"\"\"\n    out = set()\n    f = j.get(\"fields\") or []\n    idate = next((i for i, x in enumerate(f) if \"日期\" in x), None)\n    icode = next((i for i, x in enumerate(f) if \"代號\" in x), None)\n    if idate is None or icode is None:\n        return out\n    for row in j.get(\"data\", []):\n        digits = re.sub(r\"[^0-9]\", \"\", str(row[idate]))\n        if len(digits) == 7:\n            digits = str(int(digits[:3]) + 1911) + digits[3:]\n        code = str(row[icode]).strip()\n        if len(digits) == 8 and len(code) == 4 and code.isdigit():\n            out.add((digits, code))\n    return out\n\n\n@dataclass\nclass Panel:\n    dates: list\n    codes: list\n    vol: np.ndarray       # stocks x dates\n    value: np.ndarray\n    close: np.ndarray\n    event: np.ndarray     # bool\n    flows: dict           # class -> stocks x dates\n    index: np.ndarray     # dates\n\n\ndef build_panel(day_files, extra_events=None):\n    \"\"\"day_files: list of (date 'YYYYMMDD', t86_json, mi_json) in calendar order (trading days only).\n    extra_events: optional set of (YYYYMMDD, code) from TWT49U, united with the X change-sign flag.\"\"\"\n    parsed = []\n    codes = set()\n    for d, t86, mi in day_files:\n        fl, unavail = parse_t86(t86)\n        q, idx = parse_mi_index(mi)\n        parsed.append((d, (fl, unavail), q, idx))\n        codes |= set(q)\n    codes = sorted(codes)\n    ci = {c: i for i, c in enumerate(codes)}\n    N, T = len(codes), len(parsed)\n    nan = lambda: np.full((N, T), np.nan)\n    vol, value, close = nan(), nan(), nan()\n    event = np.zeros((N, T), bool)\n    flows = {k: nan() for k in T86_FIELDS}\n    index = np.full(T, np.nan)\n    for t, (d, (fl, unavail), q, idx) in enumerate(parsed):\n        index[t] = idx\n        for c, rec in q.items():\n            i = ci[c]\n            vol[i, t], value[i, t], close[i, t], event[i, t] = rec[\"vol\"], rec[\"value\"], rec[\"close\"], rec[\"event\"]\n            if rec[\"vol\"] and rec[\"vol\"] > 0:\n                f = fl.get(c)\n                for k in T86_FIELDS:\n                    # absent from the day's report = no institutional trade (0); a series the schema lacks = NaN\n                    flows[k][i, t] = math.nan if k in unavail else (0.0 if f is None else f[k])\n    x_only = int(event.sum())\n    if extra_events:\n        di = {p[0]: t for t, p in enumerate(parsed)}\n        for d, c in extra_events:\n            if d in di and c in ci:\n                event[ci[c], di[d]] = True\n    P = Panel([p[0] for p in parsed], codes, vol, value, close, event, flows, index)\n    P.event_counts = {\"x_flag\": x_only, \"union_with_twt49u\": int(event.sum())}\n    return P\n\n\ndef class_flows(P):\n    F = P.flows\n    out = {\n        \"Foreign\": F[\"foreign_ex\"] + F[\"foreign_dealer\"],\n        \"Trust\": F[\"trust\"],\n        \"DealerOwn\": F[\"dealer_own\"],\n        \"Hedge\": F[\"dealer_hedge\"],\n        \"DealerTotal\": F[\"dealer_total\"],   # amendment A2: the \"hedging included in the dealer class\" sensitivity\n    }\n    # Residual = minus all reported institutional flow. The dealer total replaces own + hedging, which it equals\n    # wherever the split is reported (checked by t86_diagnostics), and exists in every schema era.\n    out[\"Residual\"] = -(out[\"Foreign\"] + out[\"Trust\"] + F[\"dealer_total\"])\n    return out\n\n\ndef prepare(P, dev_mask, top_n=300, min_share=0.95, trail=60, post_listing=60):\n    \"\"\"Universe, flow scaling, beta, market-reference basis x, limit indicator and usable-origin masks.\"\"\"\n    N, T = P.close.shape\n    traded = np.isfinite(P.vol) & (P.vol > 0) & np.isfinite(P.close)\n    logc = np.log(P.close)\n    ret = np.full((N, T), np.nan)\n    ret[:, 1:] = logc[:, 1:] - logc[:, :-1]\n    prev_close = np.full((N, T), np.nan)\n    prev_close[:, 1:] = P.close[:, :-1]\n    limit = np.abs(P.close - prev_close) >= 0.095 * prev_close\n    limit &= np.isfinite(prev_close)\n    ri = np.full(T, np.nan)\n    ri[1:] = np.diff(np.log(P.index))\n    excl = P.event.copy()\n    excl[:, 1:] |= P.event[:, :-1]\n    # first `post_listing` traded days of a stock that starts trading after the first date\n    first = np.argmax(traded, axis=1)\n    for i in range(N):\n        if first[i] > 0:\n            excl[i, first[i]:first[i] + post_listing] = True\n    # universe on development days\n    dev = dev_mask[None, :]\n    usable_share = (traded & ~excl & dev).sum(1) / max(dev_mask.sum(), 1)\n    turnover = np.where(dev & traded, P.value, np.nan)\n    mean_turn = np.nanmean(turnover, axis=1)\n    elig = np.where((usable_share >= min_share) & np.isfinite(mean_turn))[0]\n    order = elig[np.lexsort((np.array(P.codes)[elig], -mean_turn[elig]))]\n    U = order[:top_n]\n    res = {\"universe_codes\": [P.codes[i] for i in U]}\n    # beta on development days\n    beta = np.full(N, np.nan)\n    for i in U:\n        m = dev_mask & traded[i] & ~excl[i] & ~limit[i] & np.isfinite(ret[i]) & np.isfinite(ri)\n        if m.sum() > 100:\n            X = ri[m]\n            beta[i] = float(np.dot(X - X.mean(), ret[i, m] - ret[i, m].mean()) / np.dot(X - X.mean(), X - X.mean()))\n    # adjusted log level: event-day return replaced by beta * index return; missing returns count as 0\n    adj = np.where(P.event, beta[:, None] * ri[None, :], ret)\n    adj = np.where(np.isfinite(adj), adj, 0.0)\n    level = np.cumsum(adj, axis=1)\n    logI = np.log(P.index)\n    basis = level - beta[:, None] * (logI[None, :] - logI[0])\n    x = np.full((N, T), np.nan)\n    for t in range(trail, T):\n        x[:, t] = basis[:, t] - basis[:, t - trail:t].mean(1)\n    flows = class_flows(P)\n    scale = np.nanmean(np.where(dev & traded, P.vol, np.nan), axis=1)\n    u = {k: v / scale[:, None] for k, v in flows.items()}\n    valid = traded & ~excl\n    usable = np.zeros((N, T), bool)\n    usable[:, 1:-1] = valid[:, :-2] & valid[:, 1:-1] & valid[:, 2:]\n    usable[:, 1:-1] &= np.isfinite(x[:, 1:-1]) & np.isfinite(x[:, :-2])\n    res.update({\"U\": U, \"x\": x, \"u\": u, \"limit\": limit.astype(float), \"usable\": usable, \"beta\": beta})\n    return res\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 3. Run\n# ----------------------------------------------------------------------------------------------------------------\n\ndef run_period(prep, fit_mask, eval_mask, cls=\"Foreign\", min_share=0.95, B=9999, mean_block=20, seed=SEED):\n    U = prep[\"U\"]\n    rows = []\n    for i in U:\n        if prep[\"usable\"][i][eval_mask].mean() < min_share:\n            continue\n        ell, _ = stock_losses(prep[\"u\"][cls][i], prep[\"x\"][i], prep[\"limit\"][i], prep[\"usable\"][i], fit_mask, eval_mask)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    L = np.vstack(rows)\n    D, per = panel_D(L)\n    boot = bootstrap_D(L, mean_block=mean_block, B=B, seed=seed)\n    out = decide(D, boot)\n    out.update({\"class\": cls, \"n_stocks\": int(L.shape[0]), \"share_stocks_positive\": float((per > 0).mean()),\n                \"mean_block\": mean_block, \"B\": B})\n    return out\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# 4. Pre-specified diagnostics and secondary outcomes (amendment A2, P1T_amendment_A2.md)\n#    Nothing above this block is changed by A2 except the extra DealerTotal class. The primary statistic is\n#    computed only by run_period.\n# ----------------------------------------------------------------------------------------------------------------\n\nDIAG_B = 1999\n\n\ndef panel_L(prep, fit_mask, eval_mask, cls=\"Foreign\", x=None, usable=None, min_share=0.95):\n    \"\"\"Stocks x dates loss-gain matrix, as in run_period, with optional replacement basis x and usable mask.\"\"\"\n    X = prep[\"x\"] if x is None else x\n    US = prep[\"usable\"] if usable is None else usable\n    rows = []\n    for i in prep[\"U\"]:\n        if US[i][eval_mask].mean() < min_share:\n            continue\n        ell, _ = stock_losses(prep[\"u\"][cls][i], X[i], prep[\"limit\"][i], US[i], fit_mask, eval_mask)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    return np.vstack(rows) if rows else np.zeros((0, len(eval_mask)))\n\n\ndef fast_stationary_indices(n, mean_block, rng):\n    \"\"\"Vectorized stationary-bootstrap indices (same law as stationary_indices, different random stream).\"\"\"\n    restart = rng.random(n) < 1.0 / mean_block\n    restart[0] = True\n    starts = rng.integers(n, size=n)\n    pos = np.arange(n)\n    last = np.maximum.accumulate(np.where(restart, pos, 0))\n    return (starts[last] + (pos - last)) % n\n\n\ndef bootstrap_D_fast(L, mean_block=20, B=DIAG_B, seed=SEED):\n    cols = np.where(np.isfinite(L).any(axis=0))[0]\n    Lc = L[:, cols]; rng = np.random.default_rng(seed); n = Lc.shape[1]\n    S = np.nan_to_num(Lc); C = np.isfinite(Lc).astype(float)\n    out = np.empty(B)\n    for b in range(B):\n        idx = fast_stationary_indices(n, mean_block, rng)\n        num = S[:, idx].sum(1); cnt = C[:, idx].sum(1)\n        out[b] = (num[cnt > 0] / cnt[cnt > 0]).mean()\n    return out\n\n\ndef summarize_L(L, mean_block=20, B=DIAG_B, seed=SEED):\n    if L.shape[0] == 0 or not np.isfinite(L).any():\n        return {\"D\": None, \"verdict\": \"not computable\", \"n_stocks\": int(L.shape[0])}\n    D, per = panel_D(L)\n    out = decide(D, bootstrap_D_fast(L, mean_block=mean_block, B=B, seed=seed))\n    out.update({\"n_stocks\": int(L.shape[0]), \"share_stocks_positive\": float((per > 0).mean()), \"B\": B,\n                \"mean_block\": mean_block})\n    return out\n\n\ndef residual_basis(prep, fit_mask, classes=(\"Foreign\", \"Trust\", \"DealerOwn\", \"Residual\", \"Hedge\")):\n    \"\"\"Spec 7.1. Per stock, OLS of x_t on a constant, x_{t-1} and the reported flows u_{c,t}, u_{c,t-1} of the four\n    classes and of dealer hedging, fitted on fit-period days where all are finite; returns x_t minus the fitted value\n    (NaN where a regressor is missing). Under the confound (prices reveal past flow) the residual carries no\n    information about future flow; under feedback it keeps the price news that flows respond to.\"\"\"\n    x = prep[\"x\"]; N, T = x.shape\n    xr = np.full((N, T), np.nan)\n    for i in prep[\"U\"]:\n        cols = [np.ones(T), np.r_[np.nan, x[i, :-1]]]\n        for c in classes:\n            u = prep[\"u\"][c][i]\n            cols += [u, np.r_[np.nan, u[:-1]]]\n        Xm = np.column_stack(cols)\n        ok = np.isfinite(Xm).all(1) & np.isfinite(x[i])\n        fit = ok & fit_mask\n        if fit.sum() < 50:\n            continue\n        b = np.linalg.lstsq(Xm[fit], x[i, fit], rcond=None)[0]\n        xr[i, ok] = x[i, ok] - Xm[ok] @ b\n    us = prep[\"usable\"].copy()\n    us[:, 1:] &= np.isfinite(xr[:, 1:]) & np.isfinite(xr[:, :-1])\n    us[:, 0] = False\n    return xr, us\n\n\ndef limit_free_usable(prep):\n    \"\"\"Spec 7.4: limit days and the following day dropped. An origin t is dropped if t-1, t or t+1 is a limit day or\n    the day after one.\"\"\"\n    lim = prep[\"limit\"].astype(bool)\n    bad = lim.copy(); bad[:, 1:] |= lim[:, :-1]\n    us = prep[\"usable\"].copy()\n    us[:, 1:-1] &= ~bad[:, :-2] & ~bad[:, 1:-1] & ~bad[:, 2:]\n    return us\n\n\ndef aggregate_k(prep, k, fit_mask, eval_mask, cls=\"Foreign\"):\n    \"\"\"Spec 7.4: k-day periods. Non-overlapping blocks of k trading days from the first date; flow = sum over the\n    block, basis = x on the block's last day, limit = any limit day in the block. A block is valid if every day is a\n    finite in u (missing or excluded days are NaN) and x is finite on the block's last day. A stock enters if at least\n    80% of its evaluation blocks are usable. Fit and evaluation blocks are those whose last day is in the mask.\"\"\"\n    u_all = prep[\"u\"][cls]; x = prep[\"x\"]; lim = prep[\"limit\"]\n    N, T = x.shape; nb = T // k\n    last = np.arange(nb) * k + (k - 1)\n    fit_b, ev_b = fit_mask[last], eval_mask[last]\n    rows = []\n    for i in prep[\"U\"]:\n        ub = u_all[i, :nb * k].reshape(nb, k).sum(1)               # NaN if any day missing\n        xb = x[i, last]\n        lb = lim[i, :nb * k].reshape(nb, k).max(1)\n        val = np.isfinite(ub) & np.isfinite(xb)\n        usb = np.zeros(nb, bool)\n        usb[1:-1] = val[:-2] & val[1:-1] & val[2:]\n        if usb[ev_b].mean() < 0.80:   # a block is lost if any of its days is; threshold 80% of evaluation blocks\n            continue\n        ell, _ = stock_losses(ub, xb, lb, usb, fit_b, ev_b)\n        if np.isfinite(ell).any():\n            rows.append(ell)\n    L = np.vstack(rows) if rows else np.zeros((0, nb))\n    return L, max(2, int(round(20 / k)))\n\n\ndef iv_tagged(u, x, ok, hac_lag=None):\n    \"\"\"Proposition 6.1 tagged IV for one stock: u_t = c + r u_{t-1} + w x_t + e_t, instruments (1, x_{t-1}, x_t).\n    ok[t] marks usable equations. Returns beta = (c, r, w), Newey-West HAC covariance, residuals and T.\"\"\"\n    t = np.where(ok)[0]\n    t = t[t >= 1]\n    if len(t) < 60:\n        return None\n    y = u[t]; X = np.column_stack([np.ones(len(t)), u[t - 1], x[t]]); Z = np.column_stack([np.ones(len(t)), x[t - 1], x[t]])\n    ZX = Z.T @ X\n    if abs(np.linalg.det(ZX)) < 1e-12:\n        return None\n    b = np.linalg.solve(ZX, Z.T @ y)\n    e = y - X @ b\n    n = len(t)\n    L = hac_lag if hac_lag is not None else int(np.floor(4 * (n / 100) ** (2 / 9)))\n    g = Z * e[:, None]\n    S = g.T @ g\n    for l in range(1, L + 1):\n        G = g[l:].T @ g[:-l]\n        S += (1 - l / (L + 1)) * (G + G.T)\n    Ainv = np.linalg.inv(ZX)\n    V = Ainv @ S @ Ainv.T\n    return {\"beta\": b, \"V\": V, \"resid\": e, \"T\": n, \"rows\": t}\n\n\ndef _iv_ok(prep, i, cls, mask):\n    u = prep[\"u\"][cls][i]; x = prep[\"x\"][i]\n    ok = prep[\"usable\"][i] & mask & np.isfinite(u)\n    ok[1:] &= np.isfinite(u[:-1]) & np.isfinite(x[:-1])\n    ok[0] = False\n    return ok\n\n\ndef secondary_S2(prep, fit_mask, eval_mask, classes=(\"Foreign\", \"Trust\", \"DealerOwn\")):\n    \"\"\"Spec 8, S2: tagged IV estimates (r_i, w_i) per class with HAC 95% intervals, in the fit and evaluation periods;\n    for Foreign, the share of stocks whose evaluation-period r_F lies in its fit-period 95% interval.\"\"\"\n    out = {}\n    for c in classes:\n        rec = {\"fit\": [], \"eval\": []}\n        inside = []\n        for i in prep[\"U\"]:\n            r = {}\n            for name, m in ((\"fit\", fit_mask), (\"eval\", eval_mask)):\n                est = iv_tagged(prep[\"u\"][c][i], prep[\"x\"][i], _iv_ok(prep, i, c, m))\n                if est is not None:\n                    se = np.sqrt(np.diag(est[\"V\"]))\n                    r[name] = (est[\"beta\"][1], se[1], est[\"beta\"][2], se[2])\n                    rec[name].append(r[name])\n            if c == \"Foreign\" and \"fit\" in r and \"eval\" in r:\n                rf, sf = r[\"fit\"][0], r[\"fit\"][1]\n                inside.append(abs(r[\"eval\"][0] - rf) <= 1.96 * sf)\n        summ = {}\n        for name in (\"fit\", \"eval\"):\n            a = np.array(rec[name]) if rec[name] else np.zeros((0, 4))\n            if len(a):\n                summ[name] = {\"n_stocks\": int(len(a)), \"r_median\": float(np.median(a[:, 0])),\n                              \"r_iqr\": [float(np.percentile(a[:, 0], 25)), float(np.percentile(a[:, 0], 75))],\n                              \"w_median\": float(np.median(a[:, 2])),\n                              \"w_iqr\": [float(np.percentile(a[:, 2], 25)), float(np.percentile(a[:, 2], 75))],\n                              \"share_w_positive_significant\": float(np.mean(a[:, 2] - 1.96 * a[:, 3] > 0)),\n                              \"share_w_negative_significant\": float(np.mean(a[:, 2] + 1.96 * a[:, 3] < 0))}\n        if c == \"Foreign\":\n            summ[\"share_eval_rF_in_fit_interval\"] = float(np.mean(inside)) if inside else None\n        out[c] = summ\n    return out\n\n\ndef formula_se_r(beta, resid, x, ok_fit, T_eval, S=200):\n    \"\"\"Eq. (ivvar) evaluated with fit-period estimates: gamma_x from the fit-period basis (demeaned), r, w from the\n    fit-period IV, sigma_f^2 = Var(e)/(1 + r^2). Returns the predicted standard error of r_hat for T_eval equations.\"\"\"\n    r, w = beta[1], beta[2]\n    xs = x[ok_fit]; xs = xs - xs.mean(); n = len(xs)\n    if n < 2 * S:\n        S = n // 4\n    gam = np.array([np.dot(xs[:n - s], xs[s:]) / n for s in range(S + 2)])\n    A = lambda m: sum((r ** s) * gam[s + m] for s in range(S + 1 - m))\n    Sig = np.array([[w * A(0), gam[1]], [w * A(1), gam[0]]])\n    sf2 = np.var(resid) / (1 + r * r)\n    G0 = np.array([[gam[0], gam[1]], [gam[1], gam[0]]])\n    G1 = np.array([[gam[1], gam[0]], [gam[2], gam[1]]])\n    Om = sf2 * ((1 + r * r) * G0 - r * (G1 + G1.T))\n    try:\n        Si = np.linalg.inv(Sig)\n    except np.linalg.LinAlgError:\n        return None\n    V = Si @ Om @ Si.T\n    return float(np.sqrt(max(V[0, 0], 0) / T_eval))\n\n\ndef secondary_S3(prep, fit_mask, eval_mask, B=199, mean_block=20, seed=SEED):\n    \"\"\"Spec 8, S3: for each stock, the eq. (ivvar) prediction of the evaluation-period standard error of r_F from\n    fit-period estimates, against a stationary block bootstrap standard error of r_F over evaluation equations.\n    Outcome: the median ratio across stocks, with agreement if it lies in [0.67, 1.5].\"\"\"\n    rng = np.random.default_rng(seed)\n    ratios = []\n    for i in prep[\"U\"]:\n        okf = _iv_ok(prep, i, \"Foreign\", fit_mask); oke = _iv_ok(prep, i, \"Foreign\", eval_mask)\n        f = iv_tagged(prep[\"u\"][\"Foreign\"][i], prep[\"x\"][i], okf)\n        e = iv_tagged(prep[\"u\"][\"Foreign\"][i], prep[\"x\"][i], oke)\n        if f is None or e is None:\n            continue\n        pred = formula_se_r(f[\"beta\"], f[\"resid\"], prep[\"x\"][i], okf, e[\"T\"])\n        if pred is None or not np.isfinite(pred) or pred <= 0:\n            continue\n        t = e[\"rows\"]; u = prep[\"u\"][\"Foreign\"][i]; x = prep[\"x\"][i]\n        y = u[t]; X = np.column_stack([np.ones(len(t)), u[t - 1], x[t]]); Z = np.column_stack([np.ones(len(t)), x[t - 1], x[t]])\n        bs = []\n        for _ in range(B):\n            idx = fast_stationary_indices(len(t), mean_block, rng)\n            ZX = Z[idx].T @ X[idx]\n            if abs(np.linalg.det(ZX)) < 1e-12:\n                continue\n            bs.append(np.linalg.solve(ZX, Z[idx].T @ y[idx])[1])\n        if len(bs) < B // 2:\n            continue\n        ratios.append(pred / float(np.std(bs)))\n    ratios = np.array(ratios)\n    if not len(ratios):\n        return {\"n_stocks\": 0, \"verdict\": \"not computable\"}\n    med = float(np.median(ratios))\n    return {\"n_stocks\": int(len(ratios)), \"median_ratio_formula_to_bootstrap\": med,\n            \"iqr_ratio\": [float(np.percentile(ratios, 25)), float(np.percentile(ratios, 75))],\n            \"share_stocks_in_range\": float(np.mean((ratios >= 0.67) & (ratios <= 1.5))),\n            \"agreement\": bool(0.67 <= med <= 1.5), \"B_per_stock\": B}\n\n\ndef holm(pvals):\n    \"\"\"Holm step-down adjusted p-values for a dict name -> p.\"\"\"\n    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items); out = {}; run = 0.0\n    for j, (k, p) in enumerate(items):\n        run = max(run, min(1.0, (m - j) * p)); out[k] = run\n    return out\n\n\ndef diagnostics(prep, dates, fit_mask, eval_mask, B=DIAG_B, seed=SEED):\n    \"\"\"All pre-specified diagnostics (spec 7) and secondary outcomes (spec 8) for one period. dates: 'YYYYMMDD'.\"\"\"\n    dates = np.array(dates)\n    res = {\"B\": B}\n    # 7.1 reverse-causality check\n    xr, usr = residual_basis(prep, fit_mask)\n    res[\"reverse_causality\"] = summarize_L(panel_L(prep, fit_mask, eval_mask, \"Foreign\", x=xr, usable=usr), B=B, seed=seed)\n    # 7.2 per-stock share (with the primary L)\n    L = panel_L(prep, fit_mask, eval_mask, \"Foreign\")\n    per = np.nanmean(L, axis=1)\n    res[\"per_stock\"] = {\"share_positive\": float(np.mean(per > 0)), \"median_d\": float(np.median(per)),\n                        \"quartiles_d\": [float(np.percentile(per, 25)), float(np.percentile(per, 75))]}\n    # 7.3 stability by calendar year, and Feb-Jun 2020 separately\n    years = sorted({d[:4] for d in dates[eval_mask]})\n    subs = {y: np.array([d[:4] == y for d in dates]) & eval_mask for y in years}\n    subs[\"2020-02_to_2020-06\"] = np.array([\"20200201\" <= d <= \"20200630\" for d in dates]) & eval_mask\n    stab = {}\n    for name, m in subs.items():\n        if m.sum() < 20:\n            continue\n        Ls = np.where(m[None, :], L, np.nan)\n        keep = np.isfinite(Ls).sum(1) >= 10\n        stab[name] = summarize_L(Ls[keep], B=B, seed=seed)\n    res[\"stability\"] = stab\n    # 7.4 sensitivity\n    sens = {}\n    sens[\"hedging_in_dealer_class\"] = summarize_L(panel_L(prep, fit_mask, eval_mask, \"DealerTotal\"), B=B, seed=seed)\n    sens[\"net_value_or_market_cap_scaling\"] = {\"verdict\": \"not computable\",\n                                               \"reason\": \"shares outstanding are not in the registered sources\"}\n    sens[\"limit_days_dropped\"] = summarize_L(panel_L(prep, fit_mask, eval_mask, \"Foreign\", usable=limit_free_usable(prep)), B=B, seed=seed)\n    for k in (2, 5):\n        Lk, mb = aggregate_k(prep, k, fit_mask, eval_mask)\n        sens[f\"{k}_day_periods\"] = summarize_L(Lk, mean_block=mb, B=B, seed=seed)\n    res[\"sensitivity\"] = sens\n    # 8 secondary outcomes\n    S1 = {}\n    for c in (\"Trust\", \"DealerOwn\"):\n        Lc = panel_L(prep, fit_mask, eval_mask, c)\n        S1[c] = run_period(prep, fit_mask, eval_mask, c) if Lc.shape[0] else {\"D\": None, \"verdict\": \"not computable\",\n                                                                              \"p_one_sided\": 1.0, \"n_stocks\": 0}\n    pv = {c: S1[c][\"p_one_sided\"] for c in S1}\n    res[\"S1\"] = {\"by_class\": S1, \"holm_adjusted_p\": holm(pv)}\n    res[\"S2\"] = secondary_S2(prep, fit_mask, eval_mask)\n    res[\"S3\"] = secondary_S3(prep, fit_mask, eval_mask)\n    return res\n\n\nif __name__ == \"__main__\":\n    print(\"P1T executor. Use the Colab notebook or validate_executor.py.\")\n"
assert hashlib.sha256(EXECUTOR_SRC.encode()).hexdigest() == EXECUTOR_SHA256, "executor changed"
open("p1t_executor.py", "w").write(EXECUTOR_SRC)
sys.path.insert(0, "."); import p1t_executor as ex

try:
    from google.colab import drive; drive.mount("/content/drive"); ROOT = "/content/drive/MyDrive/P1T_raw"
except Exception: ROOT = "P1T_raw"
os.makedirs(ROOT, exist_ok=True)
for sub in ("T86", "MI", "TWT49U", "FMTQIK"): os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
START, END, DEV_END = "20130102", "20181228", "20161230"
HOLDOUT_START = "20190102"
H = {"User-Agent": "Mozilla/5.0 (academic research; P1T)"}
MAN = os.path.join(ROOT, "manifest.jsonl")
def now(): return datetime.datetime.now(datetime.timezone.utc).isoformat()
def fetch(path, url, params, want_date=None):
    fn = os.path.join(ROOT, path + ".json.gz")
    if os.path.exists(fn):
        return json.loads(gzip.open(fn).read())
    for attempt in range(6):
        try:
            r = requests.get(url, params=params, headers=H, timeout=60)
            if r.status_code == 200:
                j = r.json()
                if want_date is not None and j.get("stat") == "OK" and j.get("date") != want_date:
                    raise SystemExit(f"DATE MISMATCH {path}: asked {want_date}, got {j.get('date')}")
                gzip.open(fn, "wb").write(r.content)
                with open(MAN, "a") as m:
                    m.write(json.dumps({"path": path, "url": r.url, "sha256": hashlib.sha256(r.content).hexdigest(), "utc": now()}) + "\n")
                time.sleep(3.5)
                return j
        except SystemExit: raise
        except Exception as e: print("retry", path, repr(e))
        time.sleep(30 * (attempt + 1))
    raise RuntimeError("failed " + path)

# 1. trading calendar from monthly market summaries
days = []
for y in range(2013, 2019):
    for mth in range(1, 13):
        j = fetch(f"FMTQIK/{y}{mth:02d}", "https://www.twse.com.tw/rwd/zh/afterTrading/FMTQIK",
                  {"date": f"{y}{mth:02d}01", "response": "json"})
        for row in j.get("data", []):
            dg = re.sub(r"[^0-9]", "", row[0]); d = str(int(dg[:-4]) + 1911) + dg[-4:]
            if START <= d <= END: days.append(d)
days = sorted(set(days)); assert days[-1] <= END and all(d < HOLDOUT_START for d in days)
print(len(days), "trading days", days[0], days[-1])

# 2. daily T86 and MI_INDEX
for k, d in enumerate(days):
    fetch(f"T86/{d}", "https://www.twse.com.tw/rwd/zh/fund/T86", {"date": d, "selectType": "ALLBUT0999", "response": "json"}, d)
    fetch(f"MI/{d}", "https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX", {"date": d, "type": "ALLBUT0999", "response": "json"}, d)
    if k % 50 == 0: print(k, d, now(), flush=True)

# 3. ex-rights table, month by month
ev = set()
for y in range(2013, 2019):
    for mth in range(1, 13):
        import calendar
        a = f"{y}{mth:02d}01"; b = f"{y}{mth:02d}{calendar.monthrange(y, mth)[1]:02d}"
        j = fetch(f"TWT49U/{y}{mth:02d}", "https://www.twse.com.tw/rwd/zh/exRight/TWT49U", {"startDate": a, "endDate": b, "response": "json"})
        ev |= ex.parse_twt49u(j)

# 4. exploratory confound sizing, development days only
def load(sub, d): return json.loads(gzip.open(os.path.join(ROOT, sub, d + ".json.gz")).read())
dev_days = [d for d in days if d <= DEV_END]
P = ex.build_panel([(d, load("T86", d), load("MI", d)) for d in dev_days], ev)
dev = np.ones(len(P.dates), bool)
prep = ex.prepare(P, dev)
x, uF, us = prep["x"], prep["u"]["Foreign"], prep["usable"]
cal = []
for i in prep["U"]:
    t = np.where(us[i] & np.isfinite(uF[i]))[0]; t = t[(t >= 1) & np.isfinite(uF[i, t - 1])]
    if len(t) < 200: continue
    y = uF[i, t]; ylag = uF[i, t - 1]
    r = float(np.polyfit(ylag, y, 1)[0]); eta = y - r * ylag; eta = eta - eta.mean()
    X = np.column_stack([x[i, t - 1], y, np.ones(len(t))]); b = np.linalg.lstsq(X, x[i, t], rcond=None)[0]
    e = x[i, t] - X @ b
    cal.append([r, b[0], b[1], float(e.std()), float(eta.std()), float(y.std())])
cal = np.array(cal); med = np.median(cal, axis=0)
r, a, kk, se, seta, sy = med
print("median calibration: r=%.3f a=%.3f k=%.4g sd_e=%.4g sd_eta=%.4g sd_u=%.4g (%d stocks)" % (r, a, kk, se, seta, sy, len(cal)))

def sim_D(noise=0.0, w=0.0, N=300, T=1475, T_dev=982, seed=1):
    rng = np.random.default_rng(seed); burn = 300; n = T + burn
    X = np.zeros((N, n)); Ut = np.zeros((N, n))
    for t in range(1, n):
        # price-first: x_t moves with this period's true flow; with feedback, flow responds to x_{t-1}
        Ut[:, t] = r * Ut[:, t - 1] + w * X[:, t - 1] + seta * rng.standard_normal(N)
        X[:, t] = a * X[:, t - 1] + kk * Ut[:, t] + se * rng.standard_normal(N)
    X, Ut = X[:, burn:], Ut[:, burn:]
    Ur = Ut + noise * Ut.std(axis=1, keepdims=True) * rng.standard_normal(Ut.shape)
    us_ = np.zeros((N, T), bool); us_[:, 2:-1] = True
    pr = {"U": range(N), "x": X, "u": {"Foreign": Ur}, "limit": np.zeros((N, T)), "usable": us_}
    devm = np.zeros(T, bool); devm[:T_dev] = True
    D, _ = ex.panel_D(ex.panel_L(pr, devm, ~devm, "Foreign"))
    return float(D)

noise_grid = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
D_noise = {s: sim_D(noise=s) for s in noise_grid}
# feedback grid: w = f (1 - max(a, r))/|k|, so that k w is a fraction f of the stability gap; unstable cases skipped
wunit = (1 - max(a, r)) / max(abs(kk), 1e-12)
def rho(w): return float(max(abs(np.linalg.eigvals(np.array([[a + kk * w, kk * r], [w, r]])))))
w_grid = [0.0, 0.01, 0.03, 0.1, 0.3, 0.6]
D_feedback = {f: (sim_D(w=f * wunit) if rho(f * wunit) < 0.999 else None) for f in w_grid}
res = {"label": "exploratory confound sizing, development days only; not a P1T statistic",
       "executor_sha256": EXECUTOR_SHA256, "run_utc": now(), "n_stocks_calibrated": int(len(cal)),
       "median_calibration": dict(zip(["r_F", "a", "k", "sd_e", "sd_eta", "sd_u"], map(float, med))),
       "calibration_quartiles": np.percentile(cal, [25, 75], axis=0).tolist(),
       "D_by_reporting_noise_no_feedback": D_noise, "D_by_feedback_no_noise": {"f_of_stability_gap": D_feedback, "w_unit": float(wunit)},
       "observed_validation_D": 0.04045892959202791}
json.dump(res, open("p1t_confound_sizing.json", "w"), indent=1, default=float)
with zipfile.ZipFile("p1t_confound_sizing.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write("p1t_confound_sizing.json")
import shutil; shutil.copy("p1t_confound_sizing.zip", os.path.join(ROOT, "p1t_confound_sizing.zip"))
print(json.dumps({k: v for k, v in res.items() if k.startswith("D_") or k == "median_calibration"}, indent=1, default=float))
try:
    from google.colab import files; files.download("p1t_confound_sizing.zip")
except Exception: print("saved p1t_confound_sizing.zip (also on Drive)")
